# Day 2, Session 4. How BERT Reads Text and Builds Inputs

The bag of words in Session 3 threw away word order and context. An encoder like BERT keeps
both. This session opens the tokenizer, reads what it produces, and assembles every input a
trainer needs. We stop just before training, which is tomorrow.

## Load the workshop helpers

One line fetches the helper functions we use across the workshop. Open
`workshop_utils.py` in the file browser on the left if you want to read them.

In [ ]:
!wget -q -O workshop_utils.py https://raw.githubusercontent.com/jacqpark/instats-python-workshop/main/workshop_utils.py
from workshop_utils import pull_manifesto
print('helpers loaded')

## What a BERT-based model is

BERT is a bidirectional encoder. It was pretrained by hiding words and learning to fill them in,
so it builds a vector for each word that depends on the words around it. `bank` in `river bank`
and `bank` in `central bank` get different vectors. The family includes BERT, RoBERTa, DeBERTa,
the multilingual XLM-R, and the smaller, faster DistilBERT we use here. A classification head is
a small layer bolted on top that turns those vectors into a label.

## Load the corpus (same pull as Session 3)

Pull the sentences, collapse to left and right, and turn the label into an integer id the model
can read.

In [ ]:
from google.colab import userdata
import pandas as pd

df = pull_manifesto(userdata.get('MANIFESTO_KEY'))
df = df[df['rile'].notna()].reset_index(drop=True)

# A model needs integers, not strings. This map is the one we carry into Day 3.
label2id = {'left': 0, 'right': 1}
id2label = {0: 'left', 1: 'right'}
df['label_id'] = df['rile'].map(label2id)
print(df['rile'].value_counts().to_string())
df[['text', 'rile', 'label_id']].head()

## Tokenization, from text to input_ids

A model does not read letters, it reads token ids. The tokenizer splits text into subword tokens
and looks up each one. Load the matching tokenizer, run it on one sentence, and read what comes
back.

In [ ]:
from transformers import AutoTokenizer
MODEL = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL)

enc = tokenizer('Expand public healthcare for all.')
print('input_ids     ', enc['input_ids'])
print('attention_mask', enc['attention_mask'])
print('tokens        ', tokenizer.convert_ids_to_tokens(enc['input_ids']))

## Padding, truncation, and the 512 wall

A batch has to be a rectangle, so short sentences are padded and long ones are truncated. The
attention mask marks which tokens are real and which are padding. BERT models cap at 512 tokens,
which is why long documents get chunked.

In [ ]:
batch = tokenizer(['Expand public healthcare.', 'Cut taxes and regulation to free enterprise.'],
                  padding=True, truncation=True, max_length=128)
for ids in batch['input_ids']:
    print(len(ids), ids)
print('both rows padded to the same length so they stack into a batch')

## Build the Dataset and tokenize it

Hugging Face `Dataset` is the table a trainer expects. Wrap the DataFrame, then map the tokenizer
over every row.

In [ ]:
from datasets import Dataset
ds = Dataset.from_pandas(df[['text','label_id']].rename(columns={'label_id':'labels'}))
def tok(batch): return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)
ds = ds.map(tok, batched=True)
print(ds)

## Train and test split

Hold out a test set now so tomorrow's evaluation is honest.

In [ ]:
split = ds.train_test_split(test_size=0.2, seed=42)
print('train', split['train'].num_rows, 'test', split['test'].num_rows)

## The model with a classification head, and one forward pass

Load the encoder with a fresh two-class head. It is untrained, so the numbers are meaningless for
now, but the shapes are the point. Logits become probabilities with softmax, and argmax picks the
predicted class. Tomorrow you train the head so those numbers mean something.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label={0:'left',1:'right'}, label2id=label2id)

batch = tokenizer(df['text'][:3].tolist(), return_tensors='pt', padding=True, truncation=True)
with torch.no_grad():
    logits = model(**batch).logits
probs = logits.softmax(-1)
print('logits', logits.shape)
print('probabilities', probs.round(decimals=3))
print('argmax label', [model.config.id2label[i] for i in probs.argmax(-1).tolist()])

## Ready to fine-tune

You have a tokenized Dataset, a train-test split, and a model with a head. Every ingredient a
trainer needs is on the table. Tomorrow you train it and prove it beats the baseline.